# Regularised deconvolution
Numerical reconstruction experiments. Original assessment text and execution outputs are excluded.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from scipy.ndimage import gaussian_filter
import scipy.sparse.linalg as splinalg
from scipy.optimize import brentq
from skimage.transform import resize
import os


In [ ]:
# 1a.) Read a grayscale image, normalise to [0, 1]
try:
    im = mpimg.imread('test_image.jpg')
    if im.ndim == 3:
        im = np.mean(im, axis=2) # Convert to grayscale
except FileNotFoundError:
    im = np.zeros((256, 256))
    im[64:192, 64:192] = 1.0
    im[128:160, 128:160] = 0.5

# Resize to 256x256 to ensure stability and reasonable compute times
im = resize(im, (256, 256), anti_aliasing=True)

f_true = np.float32(im)
f_true = (f_true - np.min(f_true)) / (np.max(f_true) - np.min(f_true))
w, h = f_true.shape
size_f = w * h

plt.figure(figsize=(5,5))
plt.imshow(f_true, cmap='gray', vmin=0, vmax=1)
plt.title('True Image')
plt.axis('off')
plt.colorbar()
plt.savefig('1a_true.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# 1b.) Set up a convolution mapping and generate g
sigma = 2.0
theta = 0.01 # Adjusted noise level

def A_func(f):
    return gaussian_filter(f, sigma)

np.random.seed(42)
noise = theta * np.random.randn(w, h)
g = A_func(f_true) + noise

plt.figure(figsize=(5,5))
plt.imshow(g, cmap='gray', vmin=0, vmax=1)
plt.title('Blurred and Noisy Image (g)')
plt.axis('off')
plt.colorbar()
plt.savefig('1b_blurred.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 1c.) Deconvolve using normal equations (GMRES)
alpha = 0.01

def ATA_func(f_vec, alpha_val):
    f_mat = f_vec.reshape((w, h))
    y = A_func(f_mat)
    z = A_func(y) + alpha_val * f_mat
    return z.flatten()

A_op = splinalg.LinearOperator((size_f, size_f), matvec=lambda f: ATA_func(f, alpha))
ATg = A_func(g).flatten()

class GMRESCounter:
    def __init__(self):
        self.niter = 0
    def __call__(self, rk=None):
        self.niter += 1

gmres_counter = GMRESCounter()
print("Task 1c: Running GMRES...")
f_alpha_gmres_vec, _ = splinalg.gmres(A_op, ATg, rtol=1e-5, maxiter=200, callback=gmres_counter)

f_alpha_gmres = f_alpha_gmres_vec.reshape((w, h))
print(f">>> GMRES Iterations to converge: {gmres_counter.niter}")

plt.figure(figsize=(5,5))
plt.imshow(f_alpha_gmres, cmap='gray')
plt.title('Task 1c: Deconvolved (GMRES)')
plt.axis('off')
plt.colorbar()
plt.savefig('1c_gmres.png', dpi=300, bbox_inches='tight')
plt.show()



In [ ]:

# 1d.) Deconvolve using augmented equations (LSQR)
def M_f(f_vec, alpha_val):
    f_mat = f_vec.reshape((w, h))
    Af = A_func(f_mat).flatten()
    sqrt_alpha_f = (np.sqrt(alpha_val) * f_mat).flatten()
    return np.concatenate([Af, sqrt_alpha_f])

def MT_b(b_vec, alpha_val):
    b1 = b_vec[:size_f].reshape((w, h))
    b2 = b_vec[size_f:].reshape((w, h))
    return A_func(b1).flatten() + np.sqrt(alpha_val) * b2.flatten()

A_aug_op = splinalg.LinearOperator((2*size_f, size_f), 
                                   matvec=lambda f: M_f(f, alpha), 
                                   rmatvec=lambda b: MT_b(b, alpha))

b_aug = np.concatenate([g.flatten(), np.zeros(size_f)])

print("Task 1d: Running LSQR...")
lsqrOutput = splinalg.lsqr(A_aug_op, b_aug, atol=1e-5, btol=1e-5, iter_lim=200)

f_alpha_lsqr = lsqrOutput[0].reshape((w, h))
lsqr_iters = lsqrOutput[2]
print(f">>> LSQR Iterations to converge: {lsqr_iters}")

plt.figure(figsize=(5,5))
plt.imshow(f_alpha_lsqr, cmap='gray')
plt.title('Task 1d: Deconvolved (LSQR)')
plt.axis('off')
plt.colorbar()
plt.savefig('1d_lsqr.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Task 2: Choose a regularisation parameter alpha

# --- i.) Discrepancy Principle ---
print("\nTask 2i: Finding optimal alpha using Discrepancy Principle...")
def discrepancy(alpha_val):
    A_aug_op_tmp = splinalg.LinearOperator((2*size_f, size_f), 
                                           matvec=lambda f: M_f(f, alpha_val), 
                                           rmatvec=lambda b: MT_b(b, alpha_val))
    f_sol_vec = splinalg.lsqr(A_aug_op_tmp, b_aug, iter_lim=100)[0]
    f_sol = f_sol_vec.reshape((w, h))
    residual = np.linalg.norm(A_func(f_sol) - g)**2
    return residual - (w * h * theta**2)

try:
    alpha_opt_dp = brentq(discrepancy, 1e-6, 0.5)
    print(f">>> Optimal alpha (Discrepancy Principle): {alpha_opt_dp:.6f}")
except ValueError:
    print("Brentq failed to find a root within the given bracket.")
    alpha_opt_dp = 0.01

# --- ii.) L-Curve ---
print("Task 2ii: Generating L-Curve...")
alphas_test = np.logspace(-4, 0, 10)
residuals_list = []
solution_norms_list = []

for a_val in alphas_test:
    A_aug_op_tmp = splinalg.LinearOperator((2*size_f, size_f), 
                                           matvec=lambda f: M_f(f, a_val), 
                                           rmatvec=lambda b: MT_b(b, a_val))
    f_sol_vec = splinalg.lsqr(A_aug_op_tmp, b_aug, iter_lim=100)[0]
    f_sol = f_sol_vec.reshape((w, h))
    
    residuals_list.append(np.linalg.norm(A_func(f_sol) - g))
    solution_norms_list.append(np.linalg.norm(f_sol))

A_aug_dp = splinalg.LinearOperator((2*size_f, size_f), 
                                   matvec=lambda f: M_f(f, alpha_opt_dp), 
                                   rmatvec=lambda b: MT_b(b, alpha_opt_dp))
f_dp_vec = splinalg.lsqr(A_aug_dp, b_aug, iter_lim=100)[0]
res_dp = np.linalg.norm(A_func(f_dp_vec.reshape((w, h))) - g)
norm_dp = np.linalg.norm(f_dp_vec)

plt.figure(figsize=(6,4))
plt.loglog(residuals_list, solution_norms_list, '-o', label='L-Curve points')
plt.scatter(res_dp, norm_dp, color='red', s=100, zorder=5, label=f'Chosen DP $\\alpha \\approx {alpha_opt_dp:.4f}$')

plt.xlabel('Residual Norm ||Af - g||_2')
plt.ylabel('Solution Norm ||f||_2')
plt.title('Task 2ii: L-Curve')
plt.grid(True, which="both", ls="--")
plt.legend() 
plt.savefig('2_lcurve.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Task 3a: Construct Gradient Operator D (Strict mathematical adjoint)
def D_op(f_mat):
    dx = np.zeros_like(f_mat)
    dy = np.zeros_like(f_mat)
    dx[:-1, :] = f_mat[1:, :] - f_mat[:-1, :]
    dy[:, :-1] = f_mat[:, 1:] - f_mat[:, :-1]
    return dx, dy

def DT_op(dx, dy):
    dx_t = np.zeros_like(dx)
    dy_t = np.zeros_like(dy)
    dx_t[1:, :] += dx[:-1, :]
    dx_t[:-1, :] -= dx[:-1, :]
    dy_t[:, 1:] += dy[:, :-1]
    dy_t[:, :-1] -= dy[:, :-1]
    return dx_t + dy_t

# ---------------------------------------------------------
alpha_grad = 0.005 
print(f"\nTask 3c: Testing alpha_grad = {alpha_grad} based on empirical tuning...")

# Task 3b: GMRES for Spatial Derivative
def ATA_grad_func(f_vec, alpha_val):
    f_mat = f_vec.reshape((w, h))
    term1 = A_func(A_func(f_mat))
    dx, dy = D_op(f_mat)
    term2 = alpha_val * DT_op(dx, dy)
    return (term1 + term2).flatten()

A_op_grad_gmres = splinalg.LinearOperator((size_f, size_f), matvec=lambda f: ATA_grad_func(f, alpha_grad))
f_grad_gmres_vec, _ = splinalg.gmres(A_op_grad_gmres, ATg, rtol=1e-5, maxiter=200)

f_grad_gmres = f_grad_gmres_vec.reshape((w, h))

plt.figure(figsize=(5,5))
plt.imshow(f_grad_gmres, cmap='gray')
plt.title('Task 3: Gradient Reg (GMRES)')
plt.axis('off')
plt.colorbar()
plt.savefig('3_grad_gmres.png', dpi=300, bbox_inches='tight')
plt.show()

# Task 3b: LSQR for Spatial Derivative
def M_f_grad(f_vec, alpha_val):
    f_mat = f_vec.reshape((w, h))
    Af = A_func(f_mat).flatten()
    dx, dy = D_op(f_mat)
    sqrt_alpha_Dx = (np.sqrt(alpha_val) * dx).flatten()
    sqrt_alpha_Dy = (np.sqrt(alpha_val) * dy).flatten()
    return np.concatenate([Af, sqrt_alpha_Dx, sqrt_alpha_Dy])

def MT_b_grad(b_vec, alpha_val):
    b_Af = b_vec[:size_f].reshape((w, h))
    b_Dx = b_vec[size_f:2*size_f].reshape((w, h))
    b_Dy = b_vec[2*size_f:].reshape((w, h))
    
    AT_b_Af = A_func(b_Af)
    DT_b_D = DT_op(b_Dx, b_Dy)
    return AT_b_Af.flatten() + np.sqrt(alpha_val) * DT_b_D.flatten()

A_aug_grad_op = splinalg.LinearOperator((3*size_f, size_f), 
                                        matvec=lambda f: M_f_grad(f, alpha_grad), 
                                        rmatvec=lambda b: MT_b_grad(b, alpha_grad))
b_aug_grad = np.concatenate([g.flatten(), np.zeros(2*size_f)])
lsqrOutput_grad = splinalg.lsqr(A_aug_grad_op, b_aug_grad, iter_lim=200)

f_grad_lsqr = lsqrOutput_grad[0].reshape((w, h))

plt.figure(figsize=(5,5))
plt.imshow(f_grad_lsqr, cmap='gray')
plt.title('Task 3: Gradient Reg (LSQR)')
plt.axis('off')
plt.colorbar()
plt.savefig('3_grad_lsqr.png', dpi=300, bbox_inches='tight')
plt.show()

# Calculate DP constraint for report justification
target_dp = w * h * theta**2
actual_residual = np.linalg.norm(A_func(f_grad_lsqr) - g)**2
print(f">>> Discrepancy target (N * theta^2): {target_dp:.4f}")
print(f">>> Actual squared residual for alpha_grad={alpha_grad}: {actual_residual:.4f}")


In [ ]:
# Task 4: Anisotropic filter
def compute_gamma(f_mat, T_thresh=None, pre_smooth_sigma=0.0, percentile=90):
    # Mild pre-smoothing stabilizes gamma against noise
    f_s = gaussian_filter(f_mat, sigma=pre_smooth_sigma) if pre_smooth_sigma > 0 else f_mat
    dx, dy = D_op(f_s)
    grad_norm = np.sqrt(dx**2 + dy**2)
    
    if T_thresh is None:
        # Dynamically compute T based on the gradient distribution
        T_val = float(np.percentile(grad_norm, percentile))
        T_val = max(T_val, 1e-6)
    else:
        T_val = T_thresh
        
    gamma = np.exp(-grad_norm / T_val).astype(np.float32)
    gamma = np.clip(gamma, 0.0, 1.0)
    
    return gamma, grad_norm, T_val

def M_f_aniso(f_vec, alpha_val, gamma):
    f_mat = f_vec.reshape((w, h))
    Af = A_func(f_mat).flatten()
    dx, dy = D_op(f_mat)
    
    sqrt_gamma = np.sqrt(gamma)
    sqrt_alpha_gamma_Dx = (np.sqrt(alpha_val) * sqrt_gamma * dx).flatten()
    sqrt_alpha_gamma_Dy = (np.sqrt(alpha_val) * sqrt_gamma * dy).flatten()
    return np.concatenate([Af, sqrt_alpha_gamma_Dx, sqrt_alpha_gamma_Dy])

def MT_b_aniso(b_vec, alpha_val, gamma):
    b_Af = b_vec[:size_f].reshape((w, h))
    b_Dx = b_vec[size_f:2*size_f].reshape((w, h))
    b_Dy = b_vec[2*size_f:].reshape((w, h))
    
    sqrt_gamma = np.sqrt(gamma)
    AT_b_Af = A_func(b_Af)
    DT_b_D = DT_op(sqrt_gamma * b_Dx, sqrt_gamma * b_Dy)
    return AT_b_Af.flatten() + np.sqrt(alpha_val) * DT_b_D.flatten()

# ---------------------------------------------------------
# Visualization for Task 4
# ---------------------------------------------------------
T_val_fixed = 0.05
# Ask for fixed T_thresh for the specific Task 4 requirements
gamma_initial, grad_norm_initial, _ = compute_gamma(g, T_thresh=T_val_fixed) 

plt.figure(figsize=(5,5))
plt.imshow(grad_norm_initial, cmap='gray')
plt.title('Gradient Magnitude |D g|')
plt.axis('off')
plt.colorbar()
plt.savefig('4_gradmag.png', dpi=300, bbox_inches='tight')
plt.show()

plt.figure(figsize=(5,5))
plt.imshow(gamma_initial, cmap='gray', vmin=0, vmax=1)
plt.title(f'Diffusivity $\gamma$ (T={T_val_fixed})')
plt.axis('off')
plt.colorbar()
plt.savefig('4_gamma.png', dpi=300, bbox_inches='tight')
plt.show()

A_aug_aniso_op = splinalg.LinearOperator((3*size_f, size_f), 
                                        matvec=lambda f: M_f_aniso(f, alpha_grad, gamma_initial), 
                                        rmatvec=lambda b: MT_b_aniso(b, alpha_grad, gamma_initial))

print("\nTask 4: Running LSQR with Anisotropic Regularisation...")
lsqrOutput_aniso = splinalg.lsqr(A_aug_aniso_op, b_aug_grad, iter_lim=150)

f_alpha_aniso = lsqrOutput_aniso[0].reshape((w, h))

plt.figure(figsize=(5,5))
plt.imshow(f_alpha_aniso, cmap='gray')
plt.title('Task 4: Anisotropic Reg (LSQR)')
plt.axis('off')
plt.colorbar()
plt.savefig('4_aniso_lsqr.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Task 5: Iterative deblurring with Termination Criterion
f_i = g.copy() 
max_iterations = 1000
tolerance = 1e-3 
rel_changes = [] 
T_history = [] 

print("\nTask 5: Running Iterative Deblurring (Adaptive)...")

for i in range(max_iterations):
    f_prev = f_i.copy()
    
    # [NEW] Compute gamma adaptively (T_thresh=None enables dynamic calculation)
    gamma_i, _, T_i = compute_gamma(f_i, T_thresh=None, pre_smooth_sigma=1.0, percentile=90)
    T_history.append(T_i)
    
    A_aug_iter_op = splinalg.LinearOperator((3*size_f, size_f), 
                                            matvec=lambda f: M_f_aniso(f, alpha_grad, gamma_i), 
                                            rmatvec=lambda b: MT_b_aniso(b, alpha_grad, gamma_i))
    
    # [KEPT] Warm start using x0=f_prev.flatten() for faster convergence
    lsqrOutput_iter = splinalg.lsqr(A_aug_iter_op, b_aug_grad, x0=f_prev.flatten(), iter_lim=100)
    
    # Clip only during iterations to keep matrix operations stable
    f_i = np.clip(lsqrOutput_iter[0].reshape((w, h)), 0, 1)
    
    rel_change = np.linalg.norm(f_i - f_prev) / (np.linalg.norm(f_prev) + 1e-12)
    rel_changes.append(rel_change) 
    
    print(f"  Iteration {i+1:02d}: Rel Change = {rel_change:.6e} | Adaptive T = {T_i:.4e}")
    
    if rel_change < tolerance:
        print(f">>> Converged! Loop terminated early at iteration {i+1}.")
        break
else:
    print(">>> Reached maximum iterations without fully converging.")

# 1. Plot Final Result
plt.figure(figsize=(5,5))
plt.imshow(f_i, cmap='gray')
plt.title('Task 5: Iterative Anisotropic (Final)')
plt.axis('off')
plt.colorbar()
plt.savefig('5_iterative.png', dpi=300, bbox_inches='tight')
plt.show()

# 2. Plot Convergence Line Chart
plt.figure(figsize=(6,4))
plt.plot(range(1, len(rel_changes)+1), rel_changes, marker='o', color='b', linewidth=2)
plt.yscale('log')
plt.axhline(y=tolerance, color='r', linestyle='--', label='Tolerance (1e-3)')
plt.title('Task 5: Convergence of Iterative Anisotropic Filter')
plt.xlabel('Iteration Number')
plt.ylabel('Relative Change')
plt.grid(True, ls='--')
plt.legend()
plt.savefig('5_convergence.png', dpi=300, bbox_inches='tight')
plt.show()

# 3. Plot Error Map
error_map = np.abs(f_i - f_true)
plt.figure(figsize=(5,5))
plt.imshow(error_map, cmap='hot')
plt.title('Task 5 Error Map: |Final - True|')
plt.axis('off')
plt.colorbar()
plt.savefig('5_errormap.png', dpi=300, bbox_inches='tight')
plt.show()